# Train your Read Aloud voice (Piper) — free GPU

Runs in **Google Colab** (Runtime → Change runtime type → **T4 GPU**) or Kaggle (Accelerator → GPU).

You'll upload the `dataset.zip` made by `03_package_dataset.py`, fine-tune a small voice from a base checkpoint, and download `your-voice.onnx` + `your-voice.onnx.json` to install in the app.

**Free sessions can disconnect.** Cell 2 mounts Google Drive so checkpoints survive — if you get kicked, just re-run from the top and training resumes from the last checkpoint.

> If a step breaks (package versions drift over time), the authoritative reference is Piper's TRAINING.md: https://github.com/OHF-Voice/piper1-gpl and the classic https://github.com/rhasspy/piper/blob/master/TRAINING.md

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU! Runtime -> Change runtime type -> T4 GPU'

## 2. Mount Drive (so checkpoints survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/voice-factory'
os.makedirs(WORK, exist_ok=True)
print('Working dir on Drive:', WORK)

## 3. Install Piper training tools
Takes a few minutes. The pinned versions are a known-good combo; if pip complains, see the TRAINING.md links above.

In [ ]:
!git clone -q https://github.com/rhasspy/piper
%cd /content/piper/src/python
!pip install -q --upgrade pip
!pip install -q -e .
!pip install -q piper-phonemize-cross 'pytorch-lightning~=1.9' 'torchmetrics==0.11.4'
!bash build_monotonic_align.sh
%cd /content
print('Piper training installed.')

## 4. Upload your dataset.zip
Choose the `dataset.zip` produced by `03_package_dataset.py`.

In [ ]:
import zipfile, os
from google.colab import files
os.makedirs('/content/dataset', exist_ok=True)
up = files.upload()  # pick dataset.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/dataset')
print('Extracted:', os.listdir('/content/dataset'))

## 5. Get the base checkpoint to fine-tune from
We start from the same voice the app's **Standard** voice came from (US English, lessac, medium), so training learns *your* sound quickly instead of from scratch.

Checkpoint filenames on the repo change over time — if the download 404s, browse https://huggingface.co/datasets/rhasspy/piper-checkpoints/tree/main/en/en_US/lessac/medium and paste the latest `.ckpt` URL below.

In [ ]:
CKPT_URL = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!wget -q -O /content/base.ckpt "$CKPT_URL" && echo 'Base checkpoint ready.' || echo 'Download failed — paste the current .ckpt URL (see cell notes).'

## 6. Preprocess the dataset

In [ ]:
!python -m piper_train.preprocess \
  --language en-us \
  --input-dir /content/dataset \
  --output-dir /content/training \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

## 7. Fine-tune
Checkpoints are written to Drive (`--default_root_dir`), so a disconnect is safe — re-run this cell and it continues from the newest checkpoint. Watch the loss go down. A few thousand steps is usually plenty for a fine-tune; you can stop early (interrupt) once the test in step 9 sounds good.

In [ ]:
import glob, os
os.makedirs(f'{WORK}/logs', exist_ok=True)
# Resume from the newest saved checkpoint if one exists, else from the base voice.
saved = sorted(glob.glob(f'{WORK}/logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
resume = saved[-1] if saved else '/content/base.ckpt'
print('Resuming from:', resume)
!python -m piper_train \
  --dataset-dir /content/training \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --max_epochs 6000 \
  --checkpoint-epochs 1 \
  --precision 16 \
  --default_root_dir "$WORK/logs" \
  --resume_from_checkpoint "$resume"

## 8. Export your voice to .onnx

In [ ]:
import glob, os, shutil
ckpts = sorted(glob.glob(f'{WORK}/logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
assert ckpts, 'No checkpoint found — did training run?'
last = ckpts[-1]
print('Exporting:', last)
!python -m piper_train.export_onnx "$last" /content/your-voice.onnx
shutil.copy('/content/training/config.json', '/content/your-voice.onnx.json')
print('Exported your-voice.onnx (+ .onnx.json)')

## 9. Listen to a test sentence
If it sounds like you, download in step 10. If it's rough, go back to step 7 and train longer (re-running resumes).

In [ ]:
!pip install -q piper-tts
!echo 'In the beginning God created the heavens and the earth.' | \
  piper -m /content/your-voice.onnx -f /content/test.wav
from IPython.display import Audio
Audio('/content/test.wav')

## 10. Download your voice
Install the two files in the app: **Manage Packs → Voices → Install voice from file** (select both together).

In [ ]:
from google.colab import files
# Also copy to Drive as a backup.
import shutil
shutil.copy('/content/your-voice.onnx', f'{WORK}/your-voice.onnx')
shutil.copy('/content/your-voice.onnx.json', f'{WORK}/your-voice.onnx.json')
files.download('/content/your-voice.onnx')
files.download('/content/your-voice.onnx.json')